# NB04 — product-of-experts VAE

**In:** intrinsic expression, CNA, methylation
**Out:** `artifacts/poe_vae.eqx` (or linear fallback), `data/interim/latent_posterior.parquet`
**Gate:** held-out NLL below MOFA+ on the same split (`mofa_nll - vae_nll ≥ 0`)
**Fallback:** MOFA+ (pre-declared)


In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
for cand in [cwd, *cwd.parents]:
    if (cand / "src" / "gate.py").is_file():
        sys.path.insert(0, str(cand / "src"))
        break
    nested = cand / "v2"
    if (nested / "src" / "gate.py").is_file():
        sys.path.insert(0, str(nested / "src"))
        break

from paths import ensure_src_on_path, resolve_v2_root
from gate import gate as _gate_impl
from safety import assert_safe

V2_ROOT = resolve_v2_root()
ensure_src_on_path(V2_ROOT)
REPO_ROOT = V2_ROOT.parent
RAW = V2_ROOT / "data" / "raw"
INTERIM = V2_ROOT / "data" / "interim"
REF = V2_ROOT / "data" / "reference"
ARTIFACTS = V2_ROOT / "artifacts"
FIGURES = V2_ROOT / "reports" / "figures"
for d in (RAW, INTERIM, REF, ARTIFACTS, FIGURES, INTERIM / "causal_networks"):
    d.mkdir(parents=True, exist_ok=True)

# Laptop vs VPS. Smoke passes are provisional until a full run converts them.
# NB01 and NB04 stay full: harmonisation and the VAE are cheap.
SMOKE_TEST = True
N_SAMPLES  = 200    if SMOKE_TEST else None   # NB02 bulk (BayesPrism; memory)
N_SC_CELLS = 25_000 if SMOKE_TEST else None   # NB02 Wu reference (BayesPrism; memory)
N_PATIENTS = 50     if SMOKE_TEST else None   # NB07 CARNIVAL (throughput, not RAM)
N_DRUGS    = 10     if SMOKE_TEST else None   # NB10 ODE (FLOPs, not RAM)

def gate(*args, **kwargs):
    kwargs.setdefault("smoke_test", SMOKE_TEST)
    return _gate_impl(*args, **kwargs)

print("V2_ROOT =", V2_ROOT, "SMOKE_TEST =", SMOKE_TEST)


In [ ]:
# Config
# NB04 is cheap: fit on the full latent cohort (no N_SAMPLES cap).
LATENT_DIM = 16
HELD_OUT = 0.2
SEED = 0
POST = INTERIM / "latent_posterior.parquet"
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from poe_vae import HAS_JAX, fit_linear_poe, gaussian_nll
from transforms import product_of_experts
from demo_patients import load_demo_exclude_ids
DEMO_EXCLUDE = load_demo_exclude_ids(REF / "demo_patients.json")
print("NB04 demo patients excluded from fit", DEMO_EXCLUDE)


In [ ]:
# Load views
expr_p = INTERIM / "intrinsic_expression.parquet"
harm_p = INTERIM / "harmonised_expression.parquet"
expr = None
if harm_p.exists():
    expr = pd.read_parquet(harm_p)
    print("NB04 using full harmonised matrix", expr.shape)
elif expr_p.exists():
    expr = pd.read_parquet(expr_p)

def load_view(patterns, index):
    for root in (RAW / "metabric", REPO_ROOT / "brca_metabric", RAW / "tcga_brca"):
        if not root.exists():
            continue
        for pat in patterns:
            hits = list(root.rglob(pat))
            if hits:
                df = pd.read_csv(hits[0], sep="\t", comment="#")
                if "Hugo_Symbol" in df.columns:
                    df = df.set_index("Hugo_Symbol")
                drop = [c for c in df.columns if "entrez" in c.lower()]
                df = df.drop(columns=drop, errors="ignore").apply(pd.to_numeric, errors="coerce").T
                df.index = df.index.astype(str)
                return df.reindex(index)
    return None

if expr is None:
    print("no expression view")
    views = None
else:
    expr = expr.select_dtypes(include=[np.number])
    expr.index = expr.index.astype(str)
    # top variable genes for a tractable encoder
    var = expr.var().nlargest(min(500, expr.shape[1])).index
    rna_genes = [str(c) for c in var]
    rna = expr[var].fillna(0).to_numpy()
    cna = load_view(["*data_cna.txt"], expr.index)
    meth = load_view(["*methylation*"], expr.index)
    if cna is not None:
        cna_genes = [str(c) for c in cna.var().nlargest(min(500, cna.shape[1])).index]
        cna = cna.loc[:, cna_genes].fillna(0).to_numpy()
    else:
        cna_genes = list(rna_genes)
        cna = np.zeros_like(rna)
    if meth is not None:
        meth_genes = [str(c) for c in meth.var().nlargest(min(500, meth.shape[1])).index]
        meth = meth.loc[:, meth_genes].fillna(0).to_numpy()
        meth = np.clip(meth, 0, 1)
    else:
        meth_genes = list(rna_genes)
        meth = np.clip(1 / (1 + np.exp(-rna / (rna.std() + 1e-6))), 0, 1)  # placeholder beta-like
        print("methylation missing; using logistic(RNA) placeholder so the PoE code path runs")
    views = [rna, cna, meth]
    view_genes = {"rna": rna_genes, "cna": cna_genes, "methylation": meth_genes}
    ids = expr.index.to_numpy()


In [ ]:
# Compute
vae_nll = np.inf
mofa_nll = np.inf
used = "none"
if views is not None:
    idx = np.arange(views[0].shape[0])
    demo_pref = {str(x)[:12] for x in DEMO_EXCLUDE}
    fit_idx = np.array([i for i, sid in enumerate(ids) if str(sid)[:12] not in demo_pref])
    if len(fit_idx) < 20:
        fit_idx = idx
    print("NB04 fit n=", len(fit_idx), "held-out-from-fit demo n=", int(len(idx) - len(fit_idx)))
    tr, te = train_test_split(fit_idx, test_size=HELD_OUT, random_state=SEED)
    train_views = [v[tr] for v in views]
    test_views = [v[te] for v in views]
    # MOFA+ baseline
    try:
        from mofapy2.run.entry_point import entry_point
        ent = entry_point()
        ent.set_data_matrix([[np.abs(v).astype(float)] for v in train_views], views_names=["rna", "cna", "meth"])
        ent.set_model_options(factors=LATENT_DIM)
        ent.set_train_options(iter=50, convergence_mode="fast", seed=SEED)
        ent.build(); ent.run()
        # reconstruction NLL ~ MSE under unit variance
        rec = []
        Z = np.vstack(ent.model.getExpectations()["Z"]["E"])
        # train-only; evaluate by projecting test via linear map from RNA
        from sklearn.linear_model import LinearRegression
        lr = LinearRegression().fit(train_views[0], Z)
        z_te = lr.predict(test_views[0])
        for i, v in enumerate(test_views):
            W = LinearRegression().fit(Z, train_views[i])
            hat = W.predict(z_te)
            rec.append(gaussian_nll(v, hat, np.zeros_like(hat)))
        mofa_nll = float(np.mean(rec))
        print("MOFA+ held-out NLL", mofa_nll)
    except Exception as e:
        print("MOFA+ failed, using PCA NLL", e)
        from sklearn.decomposition import PCA
        pca = PCA(min(LATENT_DIM, train_views[0].shape[0]-1)).fit(train_views[0])
        hat = pca.inverse_transform(pca.transform(test_views[0]))
        mofa_nll = gaussian_nll(test_views[0], hat, np.zeros_like(hat))

    if HAS_JAX:
        try:
            from poe_vae import train_poe_vae, PoEVAE
            import jax.numpy as jnp, jax, equinox as eqx
            model, losses = train_poe_vae(train_views, latent_dim=LATENT_DIM, steps=200, batch_size=32, seed=SEED)
            mus, lvs = [], []
            for i, v in enumerate(test_views):
                mu, lv = model.encode_view(i, jnp.asarray(v, dtype=jnp.float32))
                mus.append(np.array(mu)); lvs.append(np.array(lv))
            mask = np.ones((3, test_views[0].shape[0], 1))
            mu_j, lv_j = product_of_experts(np.stack(mus), np.stack(lvs), mask)
            recs = []
            z = mu_j
            for i, v in enumerate(test_views):
                hat = np.array(model.decode_view(i, jnp.asarray(z)))
                recs.append(gaussian_nll(v, hat, np.zeros_like(hat)))
            vae_nll = float(np.mean(recs))
            if not np.isfinite(vae_nll):
                raise RuntimeError("JAX VAE produced non-finite NLL")
            used = "jax_poe_vae"
            try:
                eqx.tree_serialise_leaves(ARTIFACTS / "poe_vae.eqx", model)
                import json as _json
                (ARTIFACTS / "poe_vae_meta.json").write_text(_json.dumps({
                    "encoder": used,
                    "latent_dim": LATENT_DIM,
                    "input_dims": [int(v.shape[1]) for v in views],
                    "genes": view_genes,
                }, indent=2))
            except Exception:
                pass
        except Exception as e:
            print("JAX VAE failed, linear PoE fallback", e)
            HAS = False
    if used == "none":
        fit = fit_linear_poe(train_views, latent_dim=LATENT_DIM)
        mu_j, lv_j = fit.encode(test_views)
        # linear reconstruct from joint mu via RNA encoder least squares
        from sklearn.linear_model import LinearRegression
        recs = []
        for i, v in enumerate(test_views):
            W = LinearRegression().fit(fit.encode(train_views)[0], train_views[i])
            recs.append(gaussian_nll(v, W.predict(mu_j), np.zeros_like(v)))
        vae_nll = float(np.mean(recs))
        used = "linear_poe"
        mu_all, lv_all = fit.encode(views)
        import json as _json
        (ARTIFACTS / "poe_vae_meta.json").write_text(_json.dumps({
            "encoder": used,
            "latent_dim": LATENT_DIM,
            "input_dims": [int(v.shape[1]) for v in views],
            "genes": view_genes,
        }, indent=2))
    else:
        # encode all samples with the trained model for NB05
        mus, lvs = [], []
        import jax.numpy as jnp
        for i, v in enumerate(views):
            mu, lv = model.encode_view(i, jnp.asarray(v, dtype=jnp.float32))
            mus.append(np.array(mu)); lvs.append(np.array(lv))
        mu_all, lv_all = product_of_experts(np.stack(mus), np.stack(lvs), np.ones((3, views[0].shape[0], 1)))

    post = pd.DataFrame(mu_all, index=ids, columns=[f"z{i}" for i in range(mu_all.shape[1])])
    post["logvar_mean"] = lv_all.mean(1)
    post["width"] = np.exp(0.5 * lv_all).mean(1)
    post["encoder"] = used
    post.to_parquet(POST)
    print("wrote", POST, "encoder", used, "vae_nll", vae_nll, "mofa_nll", mofa_nll)


In [ ]:
# GATE  (positive delta means VAE is better)
delta = (0.0 if not np.isfinite(mofa_nll) else mofa_nll) - (0.0 if not np.isfinite(vae_nll) else vae_nll)
if views is None:
    delta = -1.0
    note = "missing views"
elif not np.isfinite(vae_nll):
    delta = -1.0
    note = f"VAE NLL non-finite vs MOFA+/PCA {mofa_nll:.4f} encoder={used}"
else:
    note = f"VAE {vae_nll:.4f} vs MOFA+/PCA {mofa_nll:.4f} encoder={used}"
    if delta < 0:
        note += " | FALLBACK: continue with this posterior anyway (pre-declared MOFA+ fallback)"
gate("NB04", "vae_vs_mofa_heldout_nll", float(delta), 0.0,
     n=None if views is None else int(views[0].shape[0]),
     note=note)


In [ ]:
# Figures
try:
    import matplotlib.pyplot as plt
    if POST.exists():
        post = pd.read_parquet(POST)
        fig, ax = plt.subplots(figsize=(4, 4))
        ax.scatter(post["z0"], post["z1"], s=8, alpha=0.5)
        ax.set_title("Latent z0 vs z1")
        fig.tight_layout(); fig.savefig(FIGURES / "NB04_latent.png", dpi=140)
except Exception as e:
    print(e)
